In [ ]:
import os
import requests
import json

from pathlib import Path
from datasets import load_dataset

import sys
sys.path.append("..")

In [ ]:
data_path = Path("../../data")
ckpt_path = Path("../../ckpt")

ckpt_path.mkdir(exist_ok=True, parents=True)

warandpeace_path = data_path / "warandpeace.txt"


tokenizer_path = ckpt_path / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [ ]:
with open(warandpeace_path, 'r') as file:
    warandpeace = file.read()

In [ ]:

import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
enc_text = tokenizer.encode(warandpeace)

In [ ]:
print(len(enc_text))

In [ ]:
enc_sample = enc_text[1000: 10000]

context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: \t{x}")
print(f"y: \t\t{y}")

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, desired)
    print(tokenizer.decode(context), tokenizer.decode([desired]))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

dataset = GPTDataset(warandpeace, tokenizer, 256, 128)

In [ ]:
dataset[0]

In [ ]:
def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text, tokenizer, max_length, stride)
    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return data_loader


In [ ]:

dataloader = create_dataloader_v1(warandpeace, batch_size=8, max_length=4, stride=4, shuffle=False)

In [ ]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
inputs, targets = first_batch
print("inputs", inputs)
print("targets", targets)

In [ ]:

second_batch = next(data_iter)
inputs, targets = second_batch
print("inputs", inputs)
print("targets", targets)

In [ ]:
# encoding tokens and positions in embeddings

In [ ]:
max_length = 16
dataloader = create_dataloader_v1(warandpeace, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

In [ ]:
vocab_size = tokenizer.n_vocab
vocab_size

In [ ]:
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)



In [ ]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
inputs, targets = first_batch
print("inputs", inputs)
print("targets", targets)

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings[0])
print(token_embeddings.shape)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
pos_embeddings.shape

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
input_embeddings.shape